# Skin Segmentation：線形分類 vs 非線形分類（Google Colab）

**実データ** UCI Skin Segmentation (245,057 pixels, B/G/R, skin=1, non-skin=2)。3入力すべてを用いたモデル比較と、**Bを固定したR-G断面**の決定境界を表示します。

- ベースライン：ロジスティック回帰・線形SVM（LinearSVC）
- 非線形：決定木・RBF-SVM・Gradient Boosting（HistGradientBoosting）・ランダムフォレスト
- 正解率、Balanced accuracy、Macro F1、Skin F1、ROC-AUC、学習時間、混同行列、ROC曲線を評価
- **注意**：画素をランダム分割するため、同一元画像に由来する画素が両側に含まれうる。未知の人物・画像への汎化性能や肌色群別の公平性は本データのみでは評価できません。

Source: [UCI Skin Segmentation](https://archive.ics.uci.edu/dataset/229/skin+segmentation), Bhatt & Dhall (2009), CC BY 4.0.

---

# Skin Segmentation: Linear vs. Nonlinear Classification (Google Colab)

**Real-world dataset:** UCI Skin Segmentation (245,057 pixels, B/G/R, skin=1, non-skin=2). Compare models using all three input channels and visualize R-G decision boundaries at fixed B levels.

- Baselines: Logistic Regression and Linear SVM (LinearSVC)
- Nonlinear models: Decision Tree, RBF-SVM, Gradient Boosting (HistGradientBoosting), and Random Forest
- Metrics: Accuracy, balanced accuracy, macro F1, skin F1, ROC-AUC, training time, confusion matrices, and ROC curves
- **Caution:** Random pixel-wise splitting may put pixels from the same original image into both train and test sets. Generalization to unseen people/images and fairness across skin-tone groups cannot be established from this dataset alone.

Source: [UCI Skin Segmentation](https://archive.ics.uci.edu/dataset/229/skin+segmentation), Bhatt & Dhall (2009), CC BY 4.0.


In [ ]:
%pip -q install ucimlrepo

In [ ]:
import time, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay)
SEED = 42
raw = fetch_ucirepo(id=229)
X = raw.data.features[['R','G','B']].astype(np.float32).copy()
y_raw = raw.data.targets.iloc[:,0].astype(int)
# UCI のラベルは 1=Skin, 2=Non-Skin。評価では Skin を正例 1 に統一
assert set(y_raw.unique()) == {1,2}
y = (y_raw == 1).astype(int)
print('Data size:', X.shape, 'Skin prevalence:', f'{y.mean():.2%}')
display(X.assign(Skin=y).head())
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.25,stratify=y,random_state=SEED)


## 0. データ探索：RGB値の分布と3次元散布図

RGBの値は通常0～255です。下のヒストグラムは各色チャンネルの値の分布を **Skin / Non-Skin別** に示します。クラスのサンプル数が異なるため、縦軸は度数ではなく各クラス内で正規化した密度です。これは画素の「明るさ（輝度）」そのものではなく、**各チャンネルの強度分布**です。

---

## 0. Data exploration: RGB channel distributions and 3D scatter plot

RGB values usually range from 0 to 255. The histograms below show each channel's intensity distribution separately for **Skin and Non-Skin** pixels. Because the classes have different sample sizes, the vertical axis shows within-class normalized density rather than raw counts. These represent **individual channel intensities**, not a single measure of luminance.


In [ ]:
# RGBチャネルごとのクラス別分布（全データを使用）
fig, axs = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
colors = {'R':'#cf4040', 'G':'#31935c', 'B':'#3983cf'}
bins = np.arange(-0.5, 256.5, 4)
for ax, col in zip(axs, ['R','G','B']):
    ax.hist(X.loc[y==0,col], bins=bins, density=True, alpha=.43,
            label='Non-Skin', color='#64748b')
    ax.hist(X.loc[y==1,col], bins=bins, density=True, alpha=.56,
            label='Skin', color=colors[col])
    ax.set(title=f'{col} channel', xlabel='Intensity (0-255)', xlim=(0,255))
    ax.grid(alpha=.15)
axs[0].set_ylabel('Probability density within each class')
axs[-1].legend()
fig.suptitle('RGB intensity distributions: Skin vs Non-Skin')
plt.tight_layout()
plt.show()

print('Per-class summary (mean / median / std):')
display(X.assign(Class=np.where(y==1,'Skin','Non-Skin'))
          .groupby('Class')[['R','G','B']].agg(['mean','median','std']).round(1))


### RGB 3次元散布図（マウスで回転・拡大）

全245,057点を描くと重なるため、**各クラスから最大3,000点ずつ**ランダムに抽出して表示します。サンプリング後の点数比率は元データのクラス比率を反映しません。点の色は正解ラベル（Skin / Non-Skin）を表します。

---

### Interactive RGB 3D scatter plot (rotate and zoom with the mouse)

Plotting all 245,057 points would create heavy overlap. We therefore randomly sample **up to 3,000 pixels per class**. The sample's class proportions do not represent those of the full dataset. Point colors indicate ground-truth labels (Skin / Non-Skin).


In [ ]:
import plotly.express as px

SCATTER_N_PER_CLASS = 3000
rng = np.random.default_rng(SEED)
selected = []
for label in [0,1]:
    inds = np.flatnonzero(y.to_numpy() == label)
    selected.extend(rng.choice(inds, size=min(SCATTER_N_PER_CLASS,len(inds)), replace=False))
plot_df = X.iloc[selected].copy()
plot_df['Class'] = np.where(y.iloc[selected].to_numpy()==1, 'Skin','Non-Skin')

fig = px.scatter_3d(plot_df, x='R', y='G', z='B', color='Class',
                    color_discrete_map={'Skin':'#e58e4a','Non-Skin':'#5572b5'},
                    opacity=0.38, title='Skin Segmentation — RGB space (class-stratified sample)',
                    range_x=[0,255], range_y=[0,255], range_z=[0,255])
fig.update_traces(marker={'size':2.1})
fig.update_layout(height=660, legend_title='Actual label', scene=dict(aspectmode='cube'))
fig.show()


## 1. 全3特徴量：6モデルの比較

RBF-SVMは計算量が大きいため、**訓練データのみ層化サンプリング**して学習します。他モデルは全訓練データを使用します。テスト集合は全モデル共通です。SVMとの公平な比較を検討する場合は、下の `SVM_TRAIN_SIZE` を増やすか、全モデルを同サイズの訓練集合に揃えてください。

---

## 1. Compare six models using all three RGB features

RBF-SVM is computationally expensive, so it is trained on a **stratified subset of the training data**. Other models use the full training split; every model is evaluated on the same test split. For a fair comparison of training budgets, increase `SVM_TRAIN_SIZE` or train all models on equally sized subsets.


In [ ]:
SVM_TRAIN_SIZE = 12000  # Colab CPU向け。速度/精度に応じて増減 / Colab CPU: adjust for speed vs. accuracy
X_svm, _, y_svm, _ = train_test_split(X_train,y_train,train_size=min(SVM_TRAIN_SIZE,len(y_train)-1),
                                     stratify=y_train,random_state=SEED)
models = {
 'Logistic Regression (linear)': make_pipeline(StandardScaler(),LogisticRegression(max_iter=1200)),
 'Linear SVM': make_pipeline(StandardScaler(),LinearSVC(C=1.,max_iter=8000)),
 'Decision Tree': DecisionTreeClassifier(max_depth=10,min_samples_leaf=30,random_state=SEED),
 'RBF-SVM': make_pipeline(StandardScaler(),SVC(kernel='rbf',C=10,gamma='scale',cache_size=500)),
 'Gradient Boosting': HistGradientBoostingClassifier(max_iter=160,max_leaf_nodes=23,learning_rate=.10,
                                                     min_samples_leaf=30,random_state=SEED),
 'Random Forest': RandomForestClassifier(n_estimators=150,max_depth=18,min_samples_leaf=3,
                                        n_jobs=-1,random_state=SEED)
}
def score_model(name, model, trainX, trainy):
 t=time.perf_counter(); model.fit(trainX,trainy); elapsed=time.perf_counter()-t
 pred=model.predict(X_test)
 if hasattr(model,'predict_proba'): scores=model.predict_proba(X_test)[:,1]
 else: scores=model.decision_function(X_test)
 row={'Model':name,'Train N':len(trainy),'Accuracy':accuracy_score(y_test,pred),
      'Balanced Acc.':balanced_accuracy_score(y_test,pred),'Macro F1':f1_score(y_test,pred,average='macro'),
      'Skin F1':f1_score(y_test,pred,pos_label=1),'ROC-AUC':roc_auc_score(y_test,scores),
      'Fit sec':elapsed}
 return row,scores,pred
rows=[]; curves={}; preds={}
for name,model in models.items():
 trainX,trainy=(X_svm,y_svm) if name=='RBF-SVM' else (X_train,y_train)
 row,sc,pred=score_model(name,model,trainX,trainy)
 rows.append(row);curves[name]=sc;preds[name]=pred
results=pd.DataFrame(rows).sort_values('ROC-AUC',ascending=False)
display(results.style.format({'Accuracy':'{:.4f}','Balanced Acc.':'{:.4f}','Macro F1':'{:.4f}',
                              'Skin F1':'{:.4f}','ROC-AUC':'{:.4f}','Fit sec':'{:.2f}'}))


## 1A. 決定木の構造を可視化 / Visualize the trained decision tree

上のモデル比較で学習した **RGB 3特徴量の決定木** をそのまま表示します（再学習しません）。最初の図は講義用に深さ4までを表示したもので，木全体は下のSVGファイルでも確認できます。青はNon-Skin，オレンジはSkinです。`gini` はこのモデルで実際に使われている不純度指標であり，**情報利得（entropy）ではありません**。情報利得に基づく木に変更したい場合は，モデル定義の `criterion='entropy'` を指定して最初から学習し直してください。

---

Visualize the **already-fitted three-feature RGB decision tree** from the previous cell (without retraining). The first chart shows the top four levels for readability. The entire tree is exported as an SVG. Blue denotes Non-Skin; orange denotes Skin. This model uses **Gini impurity**, not entropy/information gain (the default criterion of scikit-learn's DecisionTreeClassifier). To experiment with information gain, set `criterion='entropy'` in the model definition and rerun training.


In [ ]:
# 既存の学習済みモデルを可視化 / Plot the previously trained tree
from sklearn.tree import plot_tree, export_graphviz
from IPython.display import SVG, display

trained_tree = models['Decision Tree']
feature_names = list(X_train.columns)  # ['R', 'G', 'B']
class_names = ['Non-Skin', 'Skin']     # y=0, y=1

print(f'Tree depth: {trained_tree.get_depth()}')
print(f'Leaves: {trained_tree.get_n_leaves()}')
print(f'Nodes: {trained_tree.tree_.node_count}')
print(f'Split criterion: {trained_tree.criterion}')

# 講義用：木の上位4階層 / Compact top four levels for teaching
fig, ax = plt.subplots(figsize=(23, 12))
plot_tree(trained_tree, feature_names=feature_names,
          class_names=class_names, filled=True, rounded=True,
          impurity=True, proportion=False,
          max_depth=3, fontsize=10, ax=ax)
ax.set_title('Skin Segmentation — Decision Tree (RGB, top 4 levels)', fontsize=17)
plt.tight_layout()
plt.savefig('skin_tree_top4.png', dpi=200, bbox_inches='tight')
plt.show()

# 木全体：Graphvizで拡大可能なSVGを作成 / Full tree as a zoomable SVG
# Colabには通常 graphviz コマンドがありますが，不足する場合は以下を実行してください：
# !apt-get -qq install graphviz
# !pip -q install graphviz
try:
    import graphviz
    dot = export_graphviz(trained_tree, out_file=None,
                          feature_names=feature_names, class_names=class_names,
                          filled=True, rounded=True, special_characters=True)
    graph = graphviz.Source(dot)
    graph.format = 'svg'
    filename = graph.render(filename='skin_tree_full', cleanup=True)
    print('Full tree exported:', filename)
    print('To view, open the SVG file from the Colab Files sidebar.')
except Exception as exc:
    print('Graphviz export skipped:', exc)
    print('Fallback: plotting the entire tree to a large PDF.')
    fig, ax = plt.subplots(figsize=(46, 22))
    plot_tree(trained_tree, feature_names=feature_names,
              class_names=class_names, filled=True, rounded=True,
              fontsize=5, ax=ax)
    plt.savefig('skin_tree_full.pdf', bbox_inches='tight')
    plt.close(fig)


In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4.5))
for name,sc in curves.items():
 RocCurveDisplay.from_predictions(y_test,sc,name=name,ax=axs[0])
axs[0].set_title('Test ROC (Skin positive)')
results.set_index('Model')['Macro F1'].sort_values().plot.barh(ax=axs[1])
axs[1].set_xlim(0,1);axs[1].set_title('Test macro F1')
plt.tight_layout();plt.show()
fig,axs=plt.subplots(2,3,figsize=(12,7))
for ax,(name,pred) in zip(axs.flat,preds.items()):
 ConfusionMatrixDisplay(confusion_matrix(y_test,pred,labels=[0,1]),display_labels=['Non-skin','Skin']).plot(ax=ax,colorbar=False,values_format='d')
 ax.set_title(name,fontsize=10)
plt.tight_layout();plt.show()


## 2. 非線形な判別境界を2次元で見る

**2変数実験**：R,Gだけを使ってモデルを再学習し、各モデルの予測確率または判定領域を表示します。各モデルを同一の部分集合で学習して比較条件を揃えます。Bを除いた2次元分類なので、上記全3変数の結果とは別の実験です。

---

## 2. Visualizing nonlinear decision boundaries in two dimensions

**Two-feature experiment:** Retrain each model using only R and G and display its prediction regions. All models use the same training subset for comparability. Because B is excluded, this is a separate experiment from the three-feature comparison above.


In [ ]:
# 判別境界の表示は高速化のために層化サンプリング（共通の訓練部分集合）
LOW_DIM_TRAIN_N = 16000
X2_train = X_train[['R','G']]
X2_test = X_test[['R','G']]
X2_sub,_,y2_sub,_=train_test_split(X2_train,y_train,train_size=LOW_DIM_TRAIN_N,
                                  stratify=y_train,random_state=SEED)
models2 = {
 'Logistic Regression':make_pipeline(StandardScaler(),LogisticRegression(max_iter=1200)),
 'Linear SVM':make_pipeline(StandardScaler(),LinearSVC(max_iter=8000)),
 'Decision Tree':DecisionTreeClassifier(max_depth=9,min_samples_leaf=30,random_state=SEED),
 'RBF-SVM':make_pipeline(StandardScaler(),SVC(kernel='rbf',C=10,gamma='scale')),
 'Gradient Boosting':HistGradientBoostingClassifier(max_iter=130,max_leaf_nodes=23,random_state=SEED),
 'Random Forest':RandomForestClassifier(n_estimators=100,max_depth=14,min_samples_leaf=5,n_jobs=-1,random_state=SEED)
}
rows2=[]
for name,m in models2.items():
 m.fit(X2_sub,y2_sub)
 p=m.predict(X2_test)
 rows2.append({'Model':name,'Accuracy':accuracy_score(y_test,p),
               'Macro F1':f1_score(y_test,p,average='macro'),
               'Balanced Acc.':balanced_accuracy_score(y_test,p)})
display(pd.DataFrame(rows2).sort_values('Macro F1',ascending=False).round(4))


In [ ]:
grid_r,grid_g=np.meshgrid(np.linspace(0,255,125),np.linspace(0,255,125))
mesh=pd.DataFrame({'R':grid_r.ravel(),'G':grid_g.ravel()})
# 画像の重なり防止のため点表示もサンプリング
plotx = X2_test.sample(n=min(2500,len(X2_test)),random_state=SEED)
ploty = y_test.loc[plotx.index]
fig,axs=plt.subplots(2,3,figsize=(15,9),sharex=True,sharey=True)
for ax,(name,m) in zip(axs.flat,models2.items()):
 zz=m.predict(mesh).reshape(grid_r.shape)
 ax.contourf(grid_r,grid_g,zz,levels=[-.5,.5,1.5],colors=['#93c5fd','#fdba74'],alpha=.36)
 ax.contour(grid_r,grid_g,zz,levels=[.5],colors='black',linewidths=1)
 ax.scatter(plotx['R'],plotx['G'],c=ploty.map({0:'#2563eb',1:'#ea580c'}),s=4,alpha=.22)
 ax.set_title(name);ax.set_xlabel('Red (R)');ax.set_ylabel('Green (G)')
plt.suptitle('2D decision regions: blue=Non-skin, orange=Skin',y=1.01)
plt.tight_layout();plt.show()


## 3. 3変数RGB空間：Bを固定した断面図

**重要**：ここで描くのは全3変数モデルを学習した後の `B=固定値` の断面です。R-Gの2変数モデルとは異なります。Bの値を変えると同じR,Gでも予測が変わる様子を観察できます。

---

## 3. Slices through the full three-dimensional RGB decision space

**Important:** These are R-G slices at a **fixed B value** of models trained using all three RGB features. They are not the same as models retrained using only R and G. Observe how the predictions change when B varies even at identical R and G coordinates.


In [ ]:
B_LEVELS = [40,110,180,240]
view_models=['Logistic Regression (linear)','Decision Tree','RBF-SVM','Gradient Boosting']
r,g=np.meshgrid(np.linspace(0,255,110),np.linspace(0,255,110))
fig,axs=plt.subplots(len(B_LEVELS),len(view_models),figsize=(14,12),sharex=True,sharey=True)
for i,b in enumerate(B_LEVELS):
 mesh3=pd.DataFrame({'R':r.ravel(),'G':g.ravel(),'B':np.full(r.size,b)})
 for j,name in enumerate(view_models):
  z=models[name].predict(mesh3).reshape(r.shape)
  ax=axs[i,j]
  ax.contourf(r,g,z,levels=[-.5,.5,1.5],colors=['#93c5fd','#fdba74'],alpha=.7)
  ax.contour(r,g,z,levels=[.5],colors='k',linewidths=.7)
  if i==0: ax.set_title(name,fontsize=10)
  if j==0: ax.set_ylabel(f'G  | B={b}')
  if i==len(B_LEVELS)-1: ax.set_xlabel('R')
plt.suptitle('Full RGB models: decision slices at fixed B',y=1.01)
plt.tight_layout();plt.show()


## 4. 写真から肌画素を抽出する実演（5種類のポートレート＋ジンジャーブレッドマン）

ここでは **学習済みの3変数RGBモデル** を、学習に用いていない人物写真へ適用します。背景や髪、衣服がどのように判定されるかも観察しましょう。

- Wikimedia Commonsの人物の公開写真5枚とジンジャーブレッドマンの写真1枚を読み込みます。人物・撮影条件・肌の見え方が異なる例として選んでいますが、**人種別公平性を評価する標本ではありません**。
- `Original`（原画像）、`Predicted mask`（推定肌領域）、`Skin pixels only`（肌と判定した部分）を表示します。
- 見やすさと処理速度のために長辺最大420ピクセルに縮小します。RGBモデルの出力を画素ごとに適用するだけで、顔検出や空間的後処理は行いません。
- **正解マスクはありません**。したがってこのデモでは各写真のIoUやDice係数を計算できず、誤分類は視覚的に確認するものです。
- 画像の二次利用については各Commonsページに表示される利用条件を確認してください。

出典： [Barack Obama](https://commons.wikimedia.org/wiki/File:President_Barack_Obama.jpg) ／ [Angela Merkel](https://commons.wikimedia.org/wiki/File:Angela_Merkel.jpg) ／ [Michelle Yeoh](https://commons.wikimedia.org/wiki/File:Michelle_Yeoh_at_San_Diego_Comic_Con_2026.jpg) ／ [Narendra Modi](https://commons.wikimedia.org/wiki/File:The_official_portrait_of_Shri_Narendra_Modi,_the_Prime_Minister_of_the_Republic_of_India.jpg) ／ [Viola Davis](https://commons.wikimedia.org/wiki/File:Viola_Davis_in_2016.jpg)

**ネガティブコントロール：** [Gingy Cookie](https://commons.wikimedia.org/wiki/File:Gingy_Cookie.jpg)（実写・CC BY-SA 3.0）。人の形と顔を持つジンジャーブレッドマンですが、もちろん人間の肌ではありません。色や見た目が肌に近い領域を、RGBだけの分類器がどの程度誤検出するかを観察します。背景も含むため、推定肌画素比率は厳密な誤検出率ではありません。

---

## 4. Skin-pixel segmentation demo on photographs (five portraits plus a gingerbread man cookie)

Apply the **trained three-feature RGB classifier** to photographs not used for training. Observe how the model classifies backgrounds, hair, and clothing.

- Fetch five public portraits and one gingerbread-man cookie photograph from Wikimedia Commons. The portraits cover varied appearances and photo conditions, but they **do not constitute a representative sample for demographic fairness evaluation**.
- Display `Original`, `Predicted mask`, and `Skin pixels only`.
- Resize images to at most 420 pixels on the longest side for speed. Classification is done pixel by pixel; no face detection or spatial post-processing is used.
- **No ground-truth masks are available**, so per-image IoU and Dice scores cannot be computed. Visually inspect the results instead.
- Check each Wikimedia Commons page for attribution and reuse requirements.

**Image sources:** [Barack Obama](https://commons.wikimedia.org/wiki/File:President_Barack_Obama.jpg) / [Angela Merkel](https://commons.wikimedia.org/wiki/File:Angela_Merkel.jpg) / [Michelle Yeoh](https://commons.wikimedia.org/wiki/File:Michelle_Yeoh_at_San_Diego_Comic_Con_2026.jpg) / [Narendra Modi](https://commons.wikimedia.org/wiki/File:The_official_portrait_of_Shri_Narendra_Modi,_the_Prime_Minister_of_the_Republic_of_India.jpg) / [Viola Davis](https://commons.wikimedia.org/wiki/File:Viola_Davis_in_2016.jpg).

**Negative control:** [Gingy Cookie](https://commons.wikimedia.org/wiki/File:Gingy_Cookie.jpg). The cookie has a human-like shape and a face, but no pixels depict actual human skin. This makes it a useful test for whether an RGB-only classifier mistakes skin-colored food for skin. The proportion predicted as skin across the whole image is not the same as a formally measured false-positive rate on annotated cookie pixels.


In [ ]:
import requests, io, urllib.parse
from PIL import Image, ImageOps
from IPython.display import display

# Wikimedia Commonsのファイルページに紐づく公開画像を使用 / Use public images from Wikimedia Commons.
# 通信エラー時は手元の写真を使用できます / If downloading fails, local files can be substituted.
DEMO_PHOTOS = [
    ('Barack Obama', 'President Barack Obama.jpg'),
    ('Angela Merkel', 'Angela Merkel.jpg'),
    ('Michelle Yeoh', 'Michelle Yeoh at San Diego Comic Con 2026.jpg'),
    ('Narendra Modi', 'The official portrait of Shri Narendra Modi, the Prime Minister of the Republic of India.jpg'),
    ('Viola Davis', 'Viola Davis in 2016.jpg'),
    ('Gingerbread man cookie (non-skin)', 'Gingy Cookie.jpg'),
]

def fetch_commons_image(filename, max_side=420):
    # APIから画像の縮小URLを取得し、Special:FilePathへの直接依存を避ける /
    # Obtain a resized image URL via the API and avoid direct dependence on Special:FilePath.
    api = 'https://commons.wikimedia.org/w/api.php'
    params = {'action':'query','format':'json','prop':'imageinfo',
              'iiprop':'url','iiurlwidth':max_side,
              'titles':'File:'+filename}
    headers = {'User-Agent':'EducationalSkinSegmentationDemo/1.0 (Google Colab example)'}
    res = requests.get(api,params=params,headers=headers,timeout=35)
    res.raise_for_status()
    page = next(iter(res.json()['query']['pages'].values()))
    info = page['imageinfo'][0]
    url = info.get('thumburl',info['url'])
    imres = requests.get(url,headers=headers,timeout=35)
    imres.raise_for_status()
    image = ImageOps.exif_transpose(Image.open(io.BytesIO(imres.content))).convert('RGB')
    image.thumbnail((max_side,max_side))
    return image

# 先に学習したmodels辞書をそのまま使用（全特徴量のモデル）
DEMO_MODEL = 'Gradient Boosting'  # 'Logistic Regression (linear)', 'Decision Tree', 'Random Forest'などに変更可能 / Switch to another model if desired
assert DEMO_MODEL in models, '先に「全3特徴量：6モデルの比較」の学習セルを実行してください。'
model = models[DEMO_MODEL]

# RGBとBGRの混同を防ぐため、列の順序を明示 / Fix the feature order explicitly to avoid RGB/BGR confusion.
FEATURE_COLS = list(X.columns)
assert set(FEATURE_COLS)=={'R','G','B'}

def segment_image(image, classifier, threshold=0.5):
    arr=np.asarray(image.convert('RGB'),dtype=np.uint8)
    rgb_flat=arr.reshape(-1,3)
    pix=pd.DataFrame(rgb_flat,columns=['R','G','B'])[FEATURE_COLS].astype(np.float32)
    if hasattr(classifier,'predict_proba'):
        mask=(classifier.predict_proba(pix)[:,1] >= threshold)
    else:
        # decision_function の場合はスコア 0 が標準の分類境界 / For decision_function, the default decision boundary is score 0.
        mask=(classifier.decision_function(pix)>=0)
    mask=mask.reshape(arr.shape[:2])
    extracted=np.full_like(arr,255)
    extracted[mask]=arr[mask]
    return mask,extracted

loaded=0
for name,filename in DEMO_PHOTOS:
    try:
        image=fetch_commons_image(filename)
        mask,extracted=segment_image(image,model)
    except Exception as exc:
        print(f'{name}: 取得または処理に失敗しました: {type(exc).__name__}: {exc}')
        continue
    fig,axes=plt.subplots(1,3,figsize=(12,4))
    axes[0].imshow(image);axes[0].set_title(name+' — original')
    axes[1].imshow(mask,cmap='gray',vmin=0,vmax=1);axes[1].set_title('Predicted skin mask')
    axes[2].imshow(extracted);axes[2].set_title('Extracted skin pixels')
    for a in axes:a.axis('off')
    fig.suptitle(f'{DEMO_MODEL}: predicted skin area {mask.mean():.1%} (not segmentation accuracy)')
    plt.tight_layout();plt.show()
    loaded+=1
print(f'Displayed {loaded} / {len(DEMO_PHOTOS)} photos.')


### 応用：手持ちの写真でも確認する（任意）

写真が取得できない場合や別の条件を試す場合、次のセルのコメントを外して実行してください。人物写真には個人情報が含まれるため、アップロードの可否に注意してください。

---

### Optional: Try your own photograph

If remote image downloading fails or you want to test another condition, uncomment and run the next cell. Photographs of people can contain personal information; consider privacy before uploading them.


In [ ]:
# 任意：Google Colabにアップロードした写真での推論
# from google.colab import files
# files_uploaded = files.upload()
# for fname, content in files_uploaded.items():
#     image = ImageOps.exif_transpose(Image.open(io.BytesIO(content))).convert('RGB')
#     image.thumbnail((420,420))
#     mask, extracted = segment_image(image, model)
#     fig, axs = plt.subplots(1,3,figsize=(12,4))
#     for ax, im, title in zip(axs,[image,mask,extracted],['Original','Predicted mask','Skin pixels only']):
#         ax.imshow(im,cmap='gray' if title=='Predicted mask' else None)
#         ax.set_title(title); ax.axis('off')
#     plt.show()


## 5. 解釈上の注意

1. まず**全3変数のテストMacro F1/ROC-AUC**で線形モデルとの差を確認する。Skinは少数クラスなのでAccuracyだけで評価しない。
2. 2D図の複雑さそれ自体は優れた汎化性能の証拠ではない。2D分類の評価値も照合する。
3. 決定木・Forest・Boostingは軸平行な分割を重ねる。RBF-SVMは滑らかな非線形境界を表現できる。
4. 色空間の変更（HSV, YCbCr等）によって線形分離性も変わる。今回は元のRGBに固定する。
5. この公開ファイルは元画像や人物のIDを持たず、独立画像に対する性能・人種別性能の検証には不十分。
6. 学習サイズの違い、とくにRBF-SVMの部分標本化には留意。さらに厳密な比較では学習数を揃え、交差検証・ハイパーパラメータ探索を訓練集合内で実施する。

**引用**：Bhatt, R., & Dhall, A. (2009), *Skin Segmentation*, UCI ML Repository, DOI: 10.24432/C5T30C.

---

## 5. Interpretation and limitations

1. First compare test-set **macro F1 and ROC-AUC** for the three-feature models. Skin is the minority class, so accuracy alone is insufficient.
2. Complex two-dimensional boundaries do not automatically imply better generalization. Check test metrics for the two-feature experiment.
3. Trees, forests, and boosting combine axis-aligned splits. RBF-SVM can describe smooth nonlinear boundaries.
4. Alternative color spaces (HSV, YCbCr, etc.) can alter linear separability. Here we deliberately use RGB.
5. The public file has neither original images nor person IDs, so independent-image generalization and demographic performance cannot be adequately assessed.
6. Note the different training sizes, particularly RBF-SVM subsampling. For a more rigorous comparison, equalize training size and perform cross-validation and hyperparameter tuning within the training split.

**Citation:** Bhatt, R., & Dhall, A. (2009), *Skin Segmentation*, UCI ML Repository, DOI: 10.24432/C5T30C.
